# 0. 开始

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import utils_z
import geopandas as gpd
import gdf_parser as gdfpar
import sql_utils

In [3]:
conn = utils_z.get_conn("Test20260413", "postgres", "we6666", "localhost", "5432")

In [70]:
city_name = "paris"
city_prefix  = "FR_PA"

# 1. gpkg文件

## 1.1 检查数据

In [13]:
test_gpkg_path = rf"E:\2_data\building_3d_opensource\{city_name}\gpkg\calgary.gpkg"

In [14]:
gdf = gpd.read_file(test_gpkg_path)

print(f"CRS: {gdf.crs}")
print(f"总行数: {len(gdf)}")
print(f"字段名: {list(gdf.columns)}")
print(f"geometry类型: {gdf.geometry.type.unique()}")
print(gdf.head())
print(gdf.describe())

CRS: PROJCS["Calgary_3TM_WGS_1984_W114",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["World_Geodetic_System_of_1984_GEM_10C",6378137,298.257223563],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-114],PARAMETER["scale_factor",0.9999],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
总行数: 457474
字段名: ['OBJECTID', 'GRD_ELEV_MAX_X', 'GRD_ELEV_MAX_Y', 'GRD_ELEV_MAX_Z', 'GRD_ELEV_MIN_X', 'GRD_ELEV_MIN_Y', 'GRD_ELEV_MIN_Z', 'ROOFTOP_ELEV_X', 'ROOFTOP_ELEV_Y', 'ROOFTOP_ELEV_Z', 'STAGE', 'STRUCT_ID', 'COLLECTION_METHODOLOGY', 'DATE_', 'GLOBALID', 'GLOBALID_GUID', 'geometry']
geometry类型: ['MultiPolygon']
   OBJECTID  GRD_ELEV_MAX_X  GRD_ELEV_MAX_Y  GRD_ELEV_MAX_Z  GRD_ELEV_MIN_X  \
0         1    -6939.542636    5.671708e+06         1120.21    -6906

In [15]:
gdf_test = gdf.copy().set_crs(3776, allow_override=True).to_crs(4326)
bounds = gdf_test.total_bounds
print(f"经度范围：{bounds[0]:.4f} ~ {bounds[2]:.4f}")
print(f"纬度范围：{bounds[1]:.4f} ~ {bounds[3]:.4f}")

经度范围：-114.2998 ~ -113.8605
纬度范围：50.8331 ~ 51.2124


In [16]:
# 取第一个建筑看geometry结构
geom = gdf.geometry.iloc[0]
print(f"polygon数量：{len(geom.geoms)}")
for i, poly in enumerate(geom.geoms):
    coords = list(poly.exterior.coords)
    z_values = [c[2] for c in coords]
    print(f"  面{i}: 顶点数={len(coords)}, Z范围={min(z_values):.2f}-{max(z_values):.2f}")

polygon数量：95
  面0: 顶点数=7, Z范围=1116.43-1124.23
  面1: 顶点数=7, Z范围=1116.43-1123.86
  面2: 顶点数=7, Z范围=1116.43-1128.25
  面3: 顶点数=7, Z范围=1127.10-1127.39
  面4: 顶点数=7, Z范围=1127.29-1128.88
  面5: 顶点数=7, Z范围=1124.62-1127.00
  面6: 顶点数=7, Z范围=1116.43-1127.00
  面7: 顶点数=7, Z范围=1124.63-1127.00
  面8: 顶点数=7, Z范围=1116.43-1127.10
  面9: 顶点数=7, Z范围=1116.43-1127.23
  面10: 顶点数=7, Z范围=1116.43-1127.65
  面11: 顶点数=7, Z范围=1126.56-1130.46
  面12: 顶点数=7, Z范围=1129.37-1130.46
  面13: 顶点数=7, Z范围=1126.66-1130.24
  面14: 顶点数=7, Z范围=1116.43-1127.23
  面15: 顶点数=7, Z范围=1116.43-1128.24
  面16: 顶点数=7, Z范围=1116.43-1124.32
  面17: 顶点数=7, Z范围=1116.43-1123.96
  面18: 顶点数=7, Z范围=1116.43-1124.27
  面19: 顶点数=7, Z范围=1116.43-1127.87
  面20: 顶点数=7, Z范围=1124.32-1127.19
  面21: 顶点数=7, Z范围=1126.71-1127.09
  面22: 顶点数=7, Z范围=1116.43-1128.59
  面23: 顶点数=7, Z范围=1116.43-1127.23
  面24: 顶点数=7, Z范围=1127.66-1129.31
  面25: 顶点数=7, Z范围=1116.43-1124.64
  面26: 顶点数=7, Z范围=1116.43-1124.32
  面27: 顶点数=7, Z范围=1116.43-1124.62
  面28: 顶点数=7, Z范围=1116.43-1129.00
  面29: 顶点数=

## 1.2 变量

In [ ]:
# lod1表和surface表的命名
block_table_name = f"block.{city_name}_blocks"

lod1_table_name_full = f"lod1.{city_name}_buildings_lod1"
lod1_surface_table_name_full = f"lod1.{city_name}_building_surfaces_lod1"

lod1_table_name = f"{city_name}_buildings_lod1"
lod1_surface_table_name = f"{city_name}_building_surfaces_lod1"

target_srid  = 4326
#source_srid  = 3857

In [5]:
# lod2表和surface表的命名
block_table_name = f"block.{city_name}_blocks"

lod2_table_name_full = f"lod2.{city_name}_buildings_lod2"
lod2_surface_table_name_full = f"lod2.{city_name}_building_surfaces_lod2"

lod2_table_name = f"{city_name}_buildings_lod2"
lod2_surface_table_name = f"{city_name}_building_surfaces_lod2"

target_srid  = 4326
source_srid  = 3776

## 1.3 建表

In [ ]:
sql_utils.create_lod2_tables(
    conn=conn, 
    lod2_table_name=lod2_table_name,
    lod2_table_name_full=lod2_table_name_full, 
    lod2_surface_table_name=lod2_surface_table_name,
    lod2_surface_table_name_full=lod2_surface_table_name_full, 
    target_srid=target_srid
)

CA_CA LOD2表创建完成


In [15]:
# 清空building和surface表，用于需要重新处理的情况
utils_z.run_sql(f"TRUNCATE TABLE {lod2_table_name_full} CASCADE;", conn=conn)
print(f"{lod2_table_name_full}表已清空")
utils_z.run_sql(f"TRUNCATE TABLE {lod2_surface_table_name_full} CASCADE;", conn=conn)
print(f"{lod2_surface_table_name_full}表已清空")

lod2.calgary_buildings_lod2表已清空
lod2.calgary_building_surfaces_lod2表已清空


## 1.4 批量入库


In [7]:
gpkg_dir = rf"E:\2_data\building_3d_opensource\{city_name}\gpkg"

In [16]:
import traceback
import glob
conn.rollback()

In [ ]:
# 查找目录下所有gpkg文件
gpkg_files = glob.glob(rf"{gpkg_dir}\*.gpkg")
print(f"找到 {len(gpkg_files)} 个gpkg文件")

# 获取当前最大ID，设置计数器初始值
cur = conn.cursor()
cur.execute(f"SELECT MAX(building_id) FROM {lod2_table_name_full}")
max_bid = cur.fetchone()[0]
building_counter = int(max_bid.split("_B_")[1]) + 1 if max_bid else 1
cur.execute(f"SELECT MAX(surface_id) FROM {lod2_surface_table_name_full}")
max_sid = cur.fetchone()[0]
surface_counter = int(max_sid.split("_S_")[1]) + 1 if max_sid else 1
cur.close()
print(f"building_counter起始值：{building_counter}")
print(f"surface_counter起始值：{surface_counter}")

total_count = 0
total_files = len(gpkg_files)

# 动态计算打印间隔（保证大约每10%的进度打印一次）
print_interval = max(1, total_files // 10)

# 依次处理每个gpkg文件
for idx, gpkg_path in enumerate(gpkg_files, 1):
    gdf = gpd.read_file(gpkg_path)

    # 分批处理
    batch_size = 50000
    total_buildings = 0
    errors = []

    for i in range(0, len(gdf), batch_size):
        gdf_batch = gdf.iloc[i:i+batch_size]
        try:
            count, building_counter, surface_counter = gdfpar.insert_buildings_gpkg_lod2(
                gdf_batch, conn,
                lod2_table=lod2_table_name_full,
                surface_table=lod2_surface_table_name_full,
                city_prefix=city_prefix,
                building_counter=building_counter,
                surface_counter=surface_counter,
                source_srid=3776,
                col_grd_z="GRD_ELEV_MIN_Z",
                col_roof_z="ROOFTOP_ELEV_Z",
                col_citygml_id="STRUCT_ID"
            )
            total_buildings += count
            print(f"批次{i//batch_size+1}/{(len(gdf)-1)//batch_size+1}完成，累计入库建筑：{total_buildings}")
        except Exception as e:
            errors.append((i, str(e)))
            print(f"批次{i//batch_size+1}错误：{e}")
            traceback.print_exc()
            conn.rollback()

print(f"\n完成！{city_name}共入库建筑：{total_buildings}，共入库surface：{surface_counter-1}")
print(f"失败批次数：{len(errors)}")

找到 1 个gpkg文件
building_counter起始值：1
surface_counter起始值：1
原始行数：49996
入库建筑：49996，跳过：0
入库surface：1760797
批次1/10完成，累计入库建筑：49996
原始行数：48784
入库建筑：48784，跳过：0
入库surface：1832670
批次2/10完成，累计入库建筑：98780
原始行数：49998
入库建筑：49998，跳过：0
入库surface：1858076
批次3/10完成，累计入库建筑：148778
原始行数：49996
入库建筑：49996，跳过：0
入库surface：999556
批次4/10完成，累计入库建筑：198774
原始行数：49317
入库建筑：49317，跳过：0
入库surface：766140
批次5/10完成，累计入库建筑：248091
原始行数：49052
入库建筑：49052，跳过：0
入库surface：712608
批次6/10完成，累计入库建筑：297143
原始行数：49995
入库建筑：49995，跳过：0
入库surface：1309762
批次7/10完成，累计入库建筑：347138
原始行数：49705
入库建筑：49705，跳过：0
入库surface：1499176
批次8/10完成，累计入库建筑：396843
原始行数：49991
入库建筑：49991，跳过：0
入库surface：1473204
批次9/10完成，累计入库建筑：446834
原始行数：7474
入库建筑：7474，跳过：0
入库surface：208793
批次10/10完成，累计入库建筑：454308

完成！calgary共入库建筑：454308，共入库surface：12420782
失败批次数：0


## 1.5 创建surface入库

In [ ]:
gdfpar.generate_surfaces_from_buildings(
    conn,
    lod1_table=lod1_table_name_full,
    surface_table=lod1_surface_table_name_full,
    city_prefix=city_prefix
)

# 统计
cur = conn.cursor()
cur.execute(f"""
    SELECT surface_type, COUNT(*) 
    FROM {lod1_surface_table_name_full} 
    GROUP BY surface_type ORDER BY surface_type
""")
rows = cur.fetchall()
for row in rows:
    print(f"  {row[0]}: {row[1]}")

# 计算总数
total = sum(row[1] for row in rows)
print(f"  Total: {total}")
cur.close()

共65986栋建筑待生成surface
已插入surface：942139  GroundSurface: 65986
  RoofSurface: 65986
  WallSurface: 813226
  GroundSurface: 65986
  RoofSurface: 65986
  WallSurface: 813226
  Total: 945198


## 1.6 与block叠合

In [ ]:
sql_utils.map_buildings_to_blocks(
    block_table_name=block_table_name,
    building_table_name=lod1_table_name,
    building_table_name_full=lod1_table_name_full,
    conn=conn
)

总建筑数：454308
成功匹配block：447301
未匹配block：7007
包含 LOD2 建筑的街区数量: 7681


# 2. Geojson

## 2.1 检查数据

In [ ]:
gdf = gpd.read_file(r"E:\2_data\building_3d_opensource\bologna\lod1_json\c_a944ctc_edifici_pl.geojson")

print(f"CRS: {gdf.crs}")
print(f"总行数: {len(gdf)}")
print(f"字段名: {list(gdf.columns)}")
print(f"geometry类型: {gdf.geometry.type.unique()}")
print(gdf.head())
print(gdf.describe())

CRS: EPSG:4326
总行数: 65781
字段名: ['geo_point_2d', 'codice_ogg', 'data_istit', 'data_varia', 'noteogg', 'perimetro', 'descrizion', 'quota_gron', 'quota_pied', 'altezza_gr', 'volume', 'origine', 'area_ogg', 'geometry']
geometry类型: ['Polygon']
                                        geo_point_2d  codice_ogg data_istit  \
0  {'lon': 11.349714738373175, 'lat': 44.48981355...       17577 2004-07-27   
1  {'lon': 11.348976811795064, 'lat': 44.48943945...       17578 2004-07-27   
2  {'lon': 11.348361366982902, 'lat': 44.51426451...        7546 2004-07-27   
3  {'lon': 11.35618095990229, 'lat': 44.536463025...        7550 2004-07-27   
4  {'lon': 11.29069769248726, 'lat': 44.510597894...       17266 2004-07-27   

  data_varia noteogg  perimetro         descrizion  quota_gron  quota_pied  \
0        NaT    None         21  Edificio generico        71.7        60.5   
1        NaT    None         22  Edificio generico        75.4        61.6   
2        NaT    None         22  Tettoia/pensilina  

## 2.2 变量

In [ ]:
# lod1表和surface表的命名
block_table_name = f"block.{city_name}_blocks"

lod1_table_name_full = f"lod1.{city_name}_buildings_lod1"
lod1_surface_table_name_full = f"lod1.{city_name}_building_surfaces_lod1"

lod1_table_name = f"{city_name}_buildings_lod1"
lod1_surface_table_name = f"{city_name}_building_surfaces_lod1"

target_srid  = 4326
source_srid  = 4326

## 2.3 建表入库

In [ ]:
sql_utils.create_lod1_tables(
    conn=conn, 
    lod2_table_name=lod1_table_name,
    lod2_table_name_full=lod1_table_name_full, 
    lod2_surface_table_name=lod1_surface_table_name,
    lod2_surface_table_name_full=lod1_surface_table_name_full, 
    target_srid=target_srid
)

IT_BO LOD1表创建完成


In [ ]:
# 清空building和surface表，用于需要重新处理的情况
utils_z.run_sql(f"TRUNCATE TABLE {lod1_table_name_full} CASCADE;", conn=conn)
print(f"{lod1_table_name_full}表已清空")
utils_z.run_sql(f"TRUNCATE TABLE {lod1_surface_table_name_full} CASCADE;", conn=conn)
print(f"{lod1_surface_table_name_full}表已清空")

lod1.bologna_building_surfaces_lod1表已清空


In [ ]:
geojson_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod1_geojson"

In [ ]:
import traceback
import glob
conn.rollback()

In [ ]:
# 查找目录下所有geojson文件
geojson_files = glob.glob(rf"{geojson_dir}\*.geojson")
print(f"找到 {len(geojson_files)} 个geojson文件")

# 获取当前最大ID，设置计数器初始值
cur = conn.cursor()
cur.execute(f"SELECT MAX(building_id) FROM {lod1_table_name_full}")
max_bid = cur.fetchone()[0]
building_counter = int(max_bid.split("_B_")[1]) + 1 if max_bid else 1
cur.close()
print(f"building_counter起始值：{building_counter}")

total_count = 0
total_files = len(geojson_files)

# 动态计算打印间隔（保证大约每10%的进度打印一次）
print_interval = max(1, total_files // 10)

# 依次处理每个geojson文件
for idx, geojson_path in enumerate(geojson_files, 1):
    try:
        gdf = gpd.read_file(geojson_path)

        # 入库
        count, building_counter = gdfpar.insert_buildings_gdf_lod1(
            gdf, conn, lod1_table=lod1_table_name_full, city_prefix=city_prefix,
            building_counter=building_counter,

            col_height="altezza_gr",
            col_area="area_ogg",
            col_ground_z="quota_pied",
            col_citygml_id="codice_ogg",
            col_function="descrizion",
            col_perimeter="perimetro",
        )
        
        
        total_count += count
        
        # 按间隔打印进度（以及最后一个文件）
        if idx % print_interval == 0 or idx == total_files:
            print(f"进度：{idx}/{total_files} 文件，已入库 {total_count} 栋建筑")
            
    except Exception as e:
        print(f"错误处理文件 {geojson_path}：{e}")
        traceback.print_exc()

print(f"\n全部完成！总共处理 {total_files} 个文件，入库 {total_count} 栋建筑")

找到 1 个geojson文件
building_counter起始值：1
进度：1/1 文件，已入库 54707 栋建筑

全部完成！总共处理 1 个文件，入库 54707 栋建筑


## 2.4 建surface

In [ ]:
gdfpar.generate_surfaces_from_buildings(
    conn,
    lod1_table=lod1_table_name_full,
    surface_table=lod1_surface_table_name_full,
    city_prefix=city_prefix
)

共54707栋建筑待生成surface
已插入surface：540355  GroundSurface: 54707
  RoofSurface: 54707
  WallSurface: 437541
  GroundSurface: 54707
  RoofSurface: 54707
  WallSurface: 437541
  Total: 546955


## 2.5 叠合block

In [ ]:
sql_utils.map_buildings_to_blocks(
    block_table_name=block_table_name,
    building_table_name=lod1_table_name,
    building_table_name_full=lod1_table_name_full,
    conn=conn
)

总建筑数：54707
成功匹配block：50968
未匹配block：3739
包含 LOD1 建筑的街区数量: 1204


# 3. WFS API

## 3.1 测试WFS链接

In [23]:
from owslib.wfs import WebFeatureService
import geopandas as gpd

In [30]:
wfs = WebFeatureService(url="http://ovc.catastro.meh.es/INSPIRE/wfsBU.aspx?", version="2.0.0")
# 获取服务描述
print("服务标题:", wfs.identification.title)
print("支持的图层:", list(wfs.contents))
# 检查 building 图层的详细信息
building_layer = wfs.contents["bu:Building"]
print("坐标系:", [crs.code for crs in building_layer.crsOptions])  # 如 ['EPSG:4490']

服务标题: Spanish INSPIRE Download Service - Buildings
支持的图层: ['bu:Building', 'bu:BuildingPart', 'bu:OtherConstruction']
坐标系: [25830, 4326, 4258, 25829, 25831, 3785, 3857, 3035, 32627, 32628, 84]


In [35]:
op = wfs.getOperationByName("GetFeature")
# op = wfs.getOperationByName("GetCapabilities")

print(op.parameters)

{}


In [41]:
response = wfs.getfeature(
    typename="bu:Building",
    bbox=(40.41, -3.71, 40.42, -3.70),
    srsname="EPSG:4326",
    maxfeatures=1
)

xml = response.read()

print(xml.decode("utf-8", errors="ignore"))

<?xml version="1.0" encoding="ISO-8859-1"?>
<!--Edificios de la D.G. del Catastro.-->
<gml:FeatureCollection gml:id="ES.SDGC.BU"  xmlns:ad="urn:x-inspire:specification:gmlas:Addresses:3.0" xmlns:base="urn:x-inspire:specification:gmlas:BaseTypes:3.2" xmlns:bu-base="http://inspire.jrc.ec.europa.eu/schemas/bu-base/3.0" xmlns:bu-core2d="http://inspire.jrc.ec.europa.eu/schemas/bu-core2d/2.0" xmlns:bu-ext2d="http://inspire.jrc.ec.europa.eu/schemas/bu-ext2d/2.0" xmlns:cp="urn:x-inspire:specification:gmlas:CadastralParcels:3.0" xmlns:el-bas="http://inspire.jrc.ec.europa.eu/schemas/el-bas/2.0" xmlns:el-cov="http://inspire.jrc.ec.europa.eu/schemas/el-cov/2.0" xmlns:el-tin="http://inspire.jrc.ec.europa.eu/schemas/el-tin/2.0" xmlns:el-vec="http://inspire.jrc.ec.europa.eu/schemas/el-vec/2.0" xmlns:gco="http://www.isotc211.org/2005/gco" xmlns:gmd="http://www.isotc211.org/2005/gmd" xmlns:gml="http://www.opengis.net/gml/3.2" xmlns:gmlcov="http://www.opengis.net/gmlcov/1.0" xmlns:gn="urn:x-inspire:spec

In [42]:
from lxml import etree

root = etree.fromstring(xml)

# 取第一个建筑对象，打印所有子元素tag
ns = {
    'bu-ext2d': 'http://inspire.jrc.ec.europa.eu/schemas/bu-ext2d/2.0',
    'bu-core2d': 'http://inspire.jrc.ec.europa.eu/schemas/bu-core2d/2.0',
    'gml': 'http://www.opengis.net/gml/3.2'
}

buildings = root.findall('.//bu-ext2d:Building', ns)
print(f"获取到建筑数：{len(buildings)}")

if buildings:
    b = buildings[0]
    print(f"\n建筑ID: {b.get('{http://www.opengis.net/gml/3.2}id')}")
    print("所有字段：")
    for child in b.iter():
        if child.text and child.text.strip():
            print(f"  {child.tag}: {child.text.strip()}")

获取到建筑数：2

建筑ID: ES.SDGC.BU.0037101VK4703E
所有字段：
  {http://www.opengis.net/gml/3.2}lowerCorner: 40.410464 -3.708863
  {http://www.opengis.net/gml/3.2}upperCorner: 40.410711 -3.708546
  {http://inspire.jrc.ec.europa.eu/schemas/bu-core2d/2.0}beginLifespanVersion: 2012-08-17T00:00:00
  {http://inspire.jrc.ec.europa.eu/schemas/bu-core2d/2.0}conditionOfConstruction: functional
  {http://inspire.jrc.ec.europa.eu/schemas/bu-core2d/2.0}beginning: 1940-01-01T00:00:00
  {http://inspire.jrc.ec.europa.eu/schemas/bu-core2d/2.0}end: 1940-01-01T00:00:00
  {http://inspire.jrc.ec.europa.eu/schemas/bu-core2d/2.0}informationSystem: https://www1.sedecatastro.gob.es/CYCBienInmueble/OVCListaBienes.aspx?rc1=0037101&rc2=VK4703E
  {http://inspire.jrc.ec.europa.eu/schemas/bu-core2d/2.0}reference: 0037101VK4703E
  {urn:x-inspire:specification:gmlas:BaseTypes:3.2}localId: 0037101VK4703E
  {urn:x-inspire:specification:gmlas:BaseTypes:3.2}namespace: ES.SDGC.BU
  {http://www.opengis.net/gml/3.2}posList: 40.410625 -3.

# 4. I3S / SLTF

In [43]:
import gzip
import json

with gzip.open(r"E:\2_data\building_3d_opensource\madrid\slpk\3dSceneLayer.json.gz", 'rb') as f:
    layer_info = json.load(f)

print(json.dumps(layer_info, indent=2)[:3000])

{
  "id": 0,
  "version": "C73609C8-EFE1-4E0B-94B0-2D470D799B44",
  "name": "MADRID_3D_2025_01",
  "serviceUpdateTimeStamp": {
    "lastUpdate": 1749636030000
  },
  "href": "./layers/0",
  "associatedLayerID": 0,
  "layerType": "3DObject",
  "spatialReference": {
    "wkid": 4326,
    "latestWkid": 4326,
    "vcsWkid": 5773,
    "latestVcsWkid": 5773
  },
  "heightModelInfo": {
    "heightModel": "gravity_related_height",
    "vertCRS": "EGM96_Geoid",
    "heightUnit": "meter"
  },
  "ZFactor": 1,
  "alias": "MADRID_3D_2025_01",
  "description": "MADRID_3D_2025_01",
  "capabilities": [
    "View",
    "Query"
  ],
  "cachedDrawingInfo": {
    "color": false
  },
  "drawingInfo": {
    "renderer": {
      "type": "simple",
      "symbol": {
        "type": "MeshSymbol3D",
        "symbolLayers": [
          {
            "type": "Fill",
            "material": {
              "color": [
                255,
                255,
                255
              ],
              "transp

# 5. 纯JSON

## 5.1 检查数据

In [67]:
# 所有字段
import json
with open(r"E:\2_data\building_3d_opensource\paris\tile_652599_6866859.json", "r", encoding="utf-8") as f:
    data = json.load(f)

if data:
    print(f"共{len(data[0].keys())}个字段：")
    for key in data[0].keys():
        val = data[0].get(key)
        print(f"  {key}: {val}")


共134个字段：
  alea_argile: Moyen
  alea_argiles: Moyen
  altitude_sol_mean: 42
  annee_construction: None
  annee_construction_dpe: None
  arrete_2021: None
  batiment_groupe_id: bdnb-bg-YYQB-NZ32-JUL4
  chauffage_solaire: None
  classe_bilan_dpe: None
  classe_conso_energie_arrete_2012: None
  classe_conso_energie_dpe_tertiaire: None
  classe_inertie: None
  cle_interop_adr_principale_ban: 75118_7665_00013
  code_commune_insee: 75118
  code_departement_insee: 75
  code_epci_insee: 200054781
  code_iris: 751187110
  code_region_insee: 11
  commune_parente: 75056
  conso_3_usages_ep_m2_arrete_2012: None
  conso_5_usages_ep_m2: None
  conso_pro_dle_elec_2020: None
  conso_pro_dle_gaz_2020: None
  conso_res_dle_elec_2020: None
  conso_res_dle_gaz_2020: None
  contient_fictive_geom_groupe: False
  contrainte_urbanisme_ac1: None
  croisement_geospx_reussi: None
  date_reception_dpe: None
  denomination_monument_historique: None
  distance_batiment_historique_plus_proche: None
  distance_monume

In [68]:
b = data[0]
print("geom_groupe:", b.get("geom_groupe"))
print("hauteur_mean:", b.get("hauteur_mean"))
print("altitude_sol_mean:", b.get("altitude_sol_mean"))
print("nb_niveau:", b.get("nb_niveau"))
print("usage_principal_bdnb_open:", b.get("usage_principal_bdnb_open"))
print("annee_construction:", b.get("annee_construction"))

geom_groupe: {'type': 'MultiPolygon', 'crs': {'type': 'name', 'properties': {'name': 'EPSG:2154'}}, 'coordinates': [[[[652826.3, 6867003], [652830.7, 6866996.2], [652826.2, 6866993.4], [652821.6, 6867000.1], [652826.3, 6867003]]], [[[652870.4, 6867011.9], [652872.9, 6866989.5], [652862.9, 6866988.5], [652860.5, 6867010.9], [652870.4, 6867011.9]]], [[[652917, 6867025.5], [652910.4, 6867025], [652909.3, 6867032.7], [652909, 6867035.4], [652908.7, 6867038], [652915.3, 6867038.4], [652915.6, 6867035.5], [652916, 6867032.8], [652917, 6867025.5]]], [[[652929.7, 6866930.5], [652931.9, 6866911.6], [652924.3, 6866910.7], [652922, 6866929.6], [652929.7, 6866930.5]]]]}
hauteur_mean: 6
altitude_sol_mean: 42
nb_niveau: None
usage_principal_bdnb_open: Tertiaire
annee_construction: None


## 5.2 变量

In [71]:
# lod1表和surface表的命名
block_table_name = f"block.{city_name}_blocks"

lod1_table_name_full = f"lod1.{city_name}_buildings_lod1"
lod1_surface_table_name_full = f"lod1.{city_name}_building_surfaces_lod1"

lod1_table_name = f"{city_name}_buildings_lod1"
lod1_surface_table_name = f"{city_name}_building_surfaces_lod1"

target_srid  = 4326
source_srid  = 2154

## 5.3 建表

In [ ]:
sql_utils.create_lod1_tables(
    conn=conn, 
    lod2_table_name=lod1_table_name,
    lod2_table_name_full=lod1_table_name_full, 
    lod2_surface_table_name=lod1_surface_table_name,
    lod2_surface_table_name_full=lod1_surface_table_name_full, 
    target_srid=target_srid
)

FR_PA LOD1表创建完成


In [73]:
# 清空building和surface表，用于需要重新处理的情况
utils_z.run_sql(f"TRUNCATE TABLE {lod1_table_name_full} CASCADE;", conn=conn)
print(f"{lod1_table_name_full}表已清空")
utils_z.run_sql(f"TRUNCATE TABLE {lod1_surface_table_name_full} CASCADE;", conn=conn)
print(f"{lod1_surface_table_name_full}表已清空")

lod1.paris_buildings_lod1表已清空
lod1.paris_building_surfaces_lod1表已清空


## 5.4 处理成GeoDataFrame并入库

In [74]:
json_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod1_json"

In [76]:
from shapely.geometry import shape

In [ ]:
# ── 入库主循环 ──────────────────────────────────────────────
json_files = [f for f in os.listdir(json_dir) if f.endswith(".json")]
print(f"共{len(json_files)}个文件待入库")

conn.rollback()
cur = conn.cursor()
cur.execute(f"SELECT MAX(building_id) FROM {lod1_table_name_full}")
max_bid = cur.fetchone()[0]
building_counter = int(max_bid.split("_B_")[1]) + 1 if max_bid else 1
cur.close()
print(f"building_counter起始值：{building_counter}")

total = 0
errors = []

for i, filename in enumerate(json_files):
    filepath = os.path.join(json_dir, filename)
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)

        if not data:
            continue

        # 转成GeoDataFrame
        rows = []
        for b in data:
            geom_raw = b.get("geom_groupe")
            if geom_raw is None:
                continue
            geom = shape(geom_raw)
            rows.append({
                "geometry":           geom,
                "hauteur_mean":       b.get("hauteur_mean"),
                "altitude_sol_mean":  b.get("altitude_sol_mean"),
                "nb_niveau":          b.get("nb_niveau"),
                "usage":              b.get("usage_principal_bdnb_open"),
                "batiment_id":        b.get("batiment_groupe_id"),
                "surface_emprise_sol":b.get("surface_emprise_sol"),
            })

        if not rows:
            continue

        gdf = gpd.GeoDataFrame(rows, crs="EPSG:2154")

        count, building_counter = gdfpar.insert_buildings_gdf_lod1(
            gdf, conn,
            lod1_table=lod1_table_name_full,
            city_prefix=city_prefix,
            building_counter=building_counter,
            col_height="hauteur_mean",
            col_ground_z="altitude_sol_mean",
            col_floor_count="nb_niveau",
            col_function="usage",
            col_citygml_id="batiment_id",
            col_area="surface_emprise_sol",
        )
        total += count

        if (i + 1) % 10 == 0:
            print(f"进度：{i+1}/{len(json_files)}，已入库建筑：{total}")

    except Exception as e:
        errors.append((filename, str(e)))
        print(f"错误：{filename} -> {e}")
        traceback.print_exc()
        conn.rollback()

print(f"\n完成！{city_name}共入库建筑：{total}")
print(f"失败文件数：{len(errors)}")

共120个文件待入库
building_counter起始值：1
过滤后建筑数：8
过滤后建筑数：9
过滤后建筑数：10
过滤后建筑数：9
过滤后建筑数：9
过滤后建筑数：10
过滤后建筑数：8
过滤后建筑数：10
过滤后建筑数：9
过滤后建筑数：8
进度：10/120，已入库建筑：90
过滤后建筑数：10
过滤后建筑数：9
过滤后建筑数：7
过滤后建筑数：10
过滤后建筑数：9
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
进度：20/120，已入库建筑：185
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：8
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：9
过滤后建筑数：7
进度：30/120，已入库建筑：279
过滤后建筑数：9
过滤后建筑数：8
过滤后建筑数：7
过滤后建筑数：10
过滤后建筑数：9
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：8
过滤后建筑数：9
进度：40/120，已入库建筑：369
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：9
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
进度：50/120，已入库建筑：468
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：9
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：8
过滤后建筑数：7
进度：60/120，已入库建筑：562
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：9
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：9
进度：70/120，已入库建筑：660
过滤后建筑数：9
过滤后建筑数：8
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：10
过滤后建筑数：8
过滤后建筑数：4
进度：80/120，已入库建筑：749
过滤后建筑数：8
过滤后建筑数：10
过滤后建筑数：7
过滤后建筑数：10

# 4. 异常排查

## 4.1 卡尔加里

In [11]:
import numpy as np
from shapely.geometry import Polygon

In [22]:
geom = gdf.geometry.iloc[0]
faces = [Polygon(list(f.exterior.coords)) for f in geom.geoms]
all_z = [c[2] for f in faces for c in f.exterior.coords]
base_z = min(all_z)

for i, face in enumerate(faces):
    coords = list(face.exterior.coords)
    pts = np.array(coords)
    v0 = pts[0]
    v1 = None
    for p in pts[1:]:
        if np.linalg.norm(p - v0) > 1e-6:
            v1 = p
            break
    if v1 is not None:
        for p in pts[1:]:
            candidate = np.cross(v1 - v0, p - v0)
            if np.linalg.norm(candidate) > 1e-6:
                normal_z = abs(candidate[2] / np.linalg.norm(candidate))
                z_mean = np.mean([c[2] for c in coords])
                print(f"面{i}: normal_z={normal_z:.4f}, z_mean={z_mean:.2f}")
                break

面0: normal_z=0.0000, z_mean=1118.61
面1: normal_z=0.0000, z_mean=1118.55
面2: normal_z=0.0000, z_mean=1119.68
面3: normal_z=0.0000, z_mean=1127.18
面4: normal_z=0.0000, z_mean=1127.54
面5: normal_z=0.0000, z_mean=1125.30
面6: normal_z=0.0000, z_mean=1119.45
面7: normal_z=0.0000, z_mean=1125.31
面8: normal_z=0.0000, z_mean=1119.46
面9: normal_z=0.0000, z_mean=1119.52
面10: normal_z=0.0000, z_mean=1119.57
面11: normal_z=0.0000, z_mean=1127.27
面12: normal_z=0.0000, z_mean=1129.65
面13: normal_z=0.0000, z_mean=1127.27
面14: normal_z=0.0000, z_mean=1119.51
面15: normal_z=0.0000, z_mean=1119.60
面16: normal_z=0.0000, z_mean=1118.63
面17: normal_z=0.0000, z_mean=1118.58
面18: normal_z=0.0000, z_mean=1118.63
面19: normal_z=0.0000, z_mean=1119.60
面20: normal_z=0.0000, z_mean=1125.13
面21: normal_z=0.0000, z_mean=1126.77
面22: normal_z=0.0000, z_mean=1119.71
面23: normal_z=0.0000, z_mean=1119.51
面24: normal_z=0.0000, z_mean=1127.91
面25: normal_z=0.0000, z_mean=1118.73
面26: normal_z=0.0000, z_mean=1118.68
面27: normal

In [ ]:
geom = gdf.geometry.iloc[0]
faces = [Polygon(list(f.exterior.coords)) for f in geom.geoms]
all_z = [c[2] for f in faces for c in f.exterior.coords if len(c) > 2]
base_z = min(all_z)
max_z = max(all_z)

for i, face in enumerate(faces):
    z_mean = np.mean([c[2] for c in face.exterior.coords if len(c) > 2])
    threshold = (max_z - base_z) * 0.2
    stype = gdfpar.classify_surfaces_flat_single(face, base_z, max_z)
    print(f"面{i}: z_mean={z_mean:.2f}, base_z={base_z:.2f}, max_z={max_z:.2f}, threshold={threshold:.2f} -> {stype}")

面0: z_mean=1118.61, base_z=1116.43, max_z=1130.86, threshold=2.89 -> GroundSurface
面1: z_mean=1118.55, base_z=1116.43, max_z=1130.86, threshold=2.89 -> GroundSurface
面2: z_mean=1119.68, base_z=1116.43, max_z=1130.86, threshold=2.89 -> WallSurface
面3: z_mean=1127.18, base_z=1116.43, max_z=1130.86, threshold=2.89 -> WallSurface
面4: z_mean=1127.54, base_z=1116.43, max_z=1130.86, threshold=2.89 -> WallSurface
面5: z_mean=1125.30, base_z=1116.43, max_z=1130.86, threshold=2.89 -> WallSurface
面6: z_mean=1119.45, base_z=1116.43, max_z=1130.86, threshold=2.89 -> WallSurface
面7: z_mean=1125.31, base_z=1116.43, max_z=1130.86, threshold=2.89 -> WallSurface
面8: z_mean=1119.46, base_z=1116.43, max_z=1130.86, threshold=2.89 -> WallSurface
面9: z_mean=1119.52, base_z=1116.43, max_z=1130.86, threshold=2.89 -> WallSurface
面10: z_mean=1119.57, base_z=1116.43, max_z=1130.86, threshold=2.89 -> WallSurface
面11: z_mean=1127.27, base_z=1116.43, max_z=1130.86, threshold=2.89 -> WallSurface
面12: z_mean=1129.65, b

In [ ]:
no_ground_count = 0
for _, row in gdf.head(1000).iterrows():
    geom = row.geometry
    if geom is None or geom.is_empty:
        continue
    faces = [Polygon(list(f.exterior.coords)) for f in geom.geoms]
    all_z = [c[2] for f in faces for c in f.exterior.coords if len(c) > 2]
    if not all_z:
        continue
    base_z = min(all_z)
    max_z = max(all_z)
    
    surfaces = {"RoofSurface": [], "WallSurface": [], "GroundSurface": []}
    for face in faces:
        stype = gdfpar.classify_surfaces_flat_single(face, base_z, max_z)
        surfaces[stype].append(face)
    
    if not surfaces["GroundSurface"]:
        no_ground_count += 1
        print(f"OBJECTID={row['OBJECTID']}: base_z={base_z:.2f}, max_z={max_z:.2f}, range={max_z-base_z:.2f}")
        z_means = [np.mean([c[2] for c in f.exterior.coords]) for f in faces]
        print(f"  z_means: {[round(z,2) for z in sorted(z_means)]}")

print(f"前1000个建筑中无底面：{no_ground_count}")

OBJECTID=931: base_z=1119.16, max_z=1133.02, range=13.86
  z_means: [np.float64(1123.44), np.float64(1123.47), np.float64(1123.5), np.float64(1123.5), np.float64(1123.51), np.float64(1123.52), np.float64(1123.54), np.float64(1123.55), np.float64(1123.56), np.float64(1123.56), np.float64(1123.59), np.float64(1123.6), np.float64(1123.6), np.float64(1123.61), np.float64(1123.64), np.float64(1123.7), np.float64(1123.72), np.float64(1123.74), np.float64(1123.75), np.float64(1123.76), np.float64(1123.78), np.float64(1123.79), np.float64(1123.81), np.float64(1123.82), np.float64(1123.82), np.float64(1123.83), np.float64(1123.83), np.float64(1123.83), np.float64(1123.87), np.float64(1123.87), np.float64(1123.87), np.float64(1123.88), np.float64(1123.89), np.float64(1123.9), np.float64(1123.91), np.float64(1123.94), np.float64(1124.29), np.float64(1124.29), np.float64(1130.54), np.float64(1130.66), np.float64(1130.93), np.float64(1130.95), np.float64(1130.96), np.float64(1131.02), np.float64(11